# AFNDetection — Colab (Arabic MSA fake-news)

**Run the cells in order** (top → bottom).

| Step | What |
|------|------|
| 1 | Mount Drive / set project path |
| 2 | Install dependencies |
| 3 | Check GPU + PyTorch (`CUDA` should be `True` on GPU runtime) |
| 4 | *(Optional)* Hugging Face token + Hub dataset check — **skip** if you use **`TRAIN_MODE = "curated"`** only |
| 5 | **Train** (pick Hub demo, Hub full, or **curated CSV** — wait for **100%** + *Writing model shards*) |
| 6 | Inference smoke test (two Arabic examples) |
| 7 | **Download ZIP** of `checkpoints/` for your PC |

**Runtime → Change runtime type → GPU** (T4 is fine). Training on CPU will be very slow.

Upload the whole **`AFNDetection`** folder to **Google Drive** (with `pyproject.toml`, `configs/train_arabic.yaml`, and `src/fndarija/` at the paths expected by the repo), then open this notebook from **Drive** in Colab.

## 1. Project folder on Drive

- Colab paths look like `/content/drive/MyDrive/...`, **not** the browser link to the folder.
- If **AFNDetection** sits at **(My Drive → AFNDetection)**, auto-detect should work. The cell below also tries common layouts such as `Deep Learning/projet/AFNDetection` (mirror of a typical coursework folder).
- If not: set **`MANUAL_ROOT`** below to your path, then re-run the cell.

In [ ]:
# @title 1 — Mount Drive & move to project
import os
import subprocess
from pathlib import Path

MODE = "drive"  # "drive" | "clone"

# If auto-detect fails, set this to your folder (must contain pyproject.toml + requirements.txt):
MANUAL_ROOT = None  # e.g. Path("/content/drive/MyDrive/AFNDetection")


def _is_project_root(d: Path) -> bool:
    return (
        d.is_dir()
        and (d / "pyproject.toml").exists()
        and (d / "requirements.txt").exists()
        and (d / "configs" / "train_arabic.yaml").exists()
        and (d / "src" / "fndarija").is_dir()
    )


def locate_project(my_drive: Path) -> Path | None:
    hint_paths = (
        Path("AFNDetection"),
        Path("DeepLearning") / "AFNDetection",
        Path("Deep Learning") / "AFNDetection",
        Path("Deep Learning") / "projet" / "AFNDetection",
        Path("S4") / "Deep Learning" / "projet" / "AFNDetection",
    )
    for rel in hint_paths:
        cand = my_drive / rel
        if _is_project_root(cand):
            return cand
    try:
        for child in my_drive.iterdir():
            if not child.is_dir():
                continue
            if _is_project_root(child):
                return child
            nested = child / "AFNDetection"
            if _is_project_root(nested):
                return nested
            deep_proj = child / "Deep Learning" / "projet" / "AFNDetection"
            if _is_project_root(deep_proj):
                return deep_proj
    except OSError:
        pass
    return None


if MODE == "drive":
    from google.colab import drive

    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    else:
        print(
            "Drive already mounted at /content/drive. "
            "To remount: drive.mount('/content/drive', force_remount=True)"
        )
    MY = Path("/content/drive/MyDrive")
    if MANUAL_ROOT is not None:
        PROJECT_ROOT = Path(MANUAL_ROOT)
        if not _is_project_root(PROJECT_ROOT):
            raise FileNotFoundError(
                f"MANUAL_ROOT is not the project root: {PROJECT_ROOT}"
            )
    else:
        PROJECT_ROOT = locate_project(MY)
        if PROJECT_ROOT is None:
            print("Top-level items in My Drive:")
            for p in sorted(MY.iterdir(), key=lambda x: x.name.lower()):
                print(" ", p.name)
            raise FileNotFoundError(
                "Project not found. Set MANUAL_ROOT = Path(\"/content/drive/MyDrive/.../AFNDetection\") "
                "and re-run."
            )
elif MODE == "clone":
    repo = "https://github.com/YOUR_USER/AFNDetection.git"  # change this
    subprocess.run(["git", "clone", repo, "/content/AFNDetection"], check=True)
    PROJECT_ROOT = Path("/content/AFNDetection")
    if not _is_project_root(PROJECT_ROOT):
        raise FileNotFoundError(
            f"Clone did not yield a valid project root (missing configs/src?): {PROJECT_ROOT}"
        )
else:
    raise ValueError('MODE must be "drive" or "clone"')

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CWD:", Path.cwd())

## 2. Install the project

Installs PyTorch (from `requirements.txt`), Hugging Face stack, Streamlit, and your package in **editable** mode so `scripts/train.py` can import `fndarija`.

If a later cell fails on `import torch` or `transformers` with a vague error, use **Runtime → Restart session**, then re-run **sections 1–2** only and continue.

In [ ]:
!pip install -q -U pip
!pip install -q -r requirements.txt
!pip install -q -e .

## 3. GPU check

You should see **`True`** and a device name (e.g. Tesla T4). If **`False`**, go back to **Runtime → Change runtime type** and select **GPU**, then **re-run from section 1**.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("⚠ Training will be very slow on CPU. Enable a GPU runtime.")

## 4. *(Optional)* Hugging Face token + Hub check

- **Skip** this cell if you only train with **`configs/train_arabic_curated.yaml`** (local CSV under `data/raw/`, no Hub dataset download).
- Otherwise it loads **`HF_TOKEN`** from Colab **Secrets** (if present) and runs `scripts/validate_datasets.py`. A non-zero exit prints a warning but **does not stop** the notebook — fix connectivity or use curated training.

In [ ]:
# @title 4 — HF token (Secrets) + validate Hub dataset
import os
import subprocess
import sys

try:
    from google.colab import userdata

    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("No HF_TOKEN secret (optional). Public Hub data usually still works.")
except Exception as exc:
    print("Colab Secrets / userdata not available:", exc)

print()
r = subprocess.run([sys.executable, "scripts/validate_datasets.py"], cwd=os.getcwd())
if r.returncode != 0:
    print(
        "\n⚠ validate_datasets.py failed — use offline-style training:\n"
        "   set TRAIN_MODE = \"curated\" in the next section, or fix Hub access / token."
    )

## 5. Train the Arabic classifier

Set **`TRAIN_MODE`** in the next cell:

| Mode | Config | Use case |
|------|--------|----------|
| `hub_demo` | `train_arabic.yaml` + 8k samples, 1 epoch | Fast sanity check (needs Hub + internet) |
| `hub_full` | `train_arabic.yaml` (defaults in YAML) | Best quality; long run |
| `curated` | `train_arabic_curated.yaml` | **Local CSV** `data/raw/arabic_curated.csv` — run `materialize_arabic_curated.py` first; good if Hub fails or for a small reproducible set |

- **First Hub run** downloads data + base model (several GB). Keep the session alive until the end.
- **Speed-up on NVIDIA GPU:** in the YAML you use, set `training.fp16: true` (leave `false` if you see NaNs).

**Normal log noise:** “unauthenticated Hub”, **LOAD REPORT** (MISSING `classifier`) when starting from AraBERT, `warmup_ratio` deprecation, long LayerNorm key messages **after** save — if the progress bar hits **100%** and you see **Writing model shards**, you are fine.

**Output folder:** `checkpoints/arabic/` (synced to Drive if the project lives on Drive).

In [ ]:
# @title 5 — Train (edit TRAIN_MODE)
import subprocess
import sys
from pathlib import Path

TRAIN_MODE = "hub_demo"  # "hub_demo" | "hub_full" | "curated"

root = Path.cwd()
if TRAIN_MODE == "hub_demo":
    cmd = [
        sys.executable,
        "scripts/train.py",
        "--config",
        "configs/train_arabic.yaml",
        "--max-samples",
        "8000",
        "--epochs",
        "1",
    ]
elif TRAIN_MODE == "hub_full":
    cmd = [sys.executable, "scripts/train.py", "--config", "configs/train_arabic.yaml"]
elif TRAIN_MODE == "curated":
    subprocess.run(
        [sys.executable, "scripts/materialize_arabic_curated.py"],
        cwd=root,
        check=True,
    )
    cmd = [sys.executable, "scripts/train.py", "--config", "configs/train_arabic_curated.yaml"]
else:
    raise ValueError(f"Unknown TRAIN_MODE: {TRAIN_MODE!r}")

print("CWD:", root)
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=root, check=True)

## 6. Inference smoke test (no Streamlit)

Loads `checkpoints/arabic` and prints **fake vs real** for a short neutral line and a sensational line (expectations depend on how you trained — a **demo** model may still mispredict).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fndarija.data.preprocess import normalize_arabic_text
from fndarija.inference.predict import load_classifier, predict_text

ckpt = ROOT / "checkpoints" / "arabic"
if not (ckpt / "config.json").exists():
    raise FileNotFoundError(
        f"No checkpoint at {ckpt}. Finish training (section 5) first."
    )

tokenizer, model, device, _ = load_classifier(ckpt)
max_len = min(getattr(model.config, "max_position_embeddings", 256), 256)
print("Using device:", device)

samples = [
    "أعلنت الوزارة في بلاغ رسمي أن الامتحانات ستنطلق وفق الجدولة المعتمدة.",
    "عاجل: مشروب واحد يشفي كل الأمراض خلال أسبوع والأطباء يخفون السر — انشر قبل الحذف!",
]

for sample in samples:
    sample = normalize_arabic_text(sample)
    label, probs, _ = predict_text(sample, tokenizer, model, device, max_length=max_len)
    print()
    print("Text:", sample[:120] + ("…" if len(sample) > 120 else ""))
    print("Label:", label, "| P(fake), P(real):", [round(float(probs[0]), 4), round(float(probs[1]), 4)])

## 7. ZIP checkpoints & download

Builds a ZIP of the whole **`checkpoints/`** tree (including `checkpoint-*` subfolders). On your PC:

1. Unzip inside your **AFNDetection** repo so you get **`checkpoints/arabic/...`**.
2. Set **`arabic_checkpoint: "checkpoints/arabic"`** in **`configs/app.yaml`** (default).
3. Install and run Streamlit from the repo root (`requirements.txt`, `pip install -e .`, then `python -m streamlit run app/streamlit_app.py`). On **PowerShell**, chain commands with **`;`** instead of `&&` if needed.

This cell **fails on purpose** if weights are missing (training not finished or Drive still syncing large files).

In [ ]:
from google.colab import files
import os
import shutil
from pathlib import Path

ROOT = Path(os.getcwd())
ckpt_root = ROOT / "checkpoints"
if not ckpt_root.is_dir():
    raise FileNotFoundError(
        f"No {ckpt_root}. Run section 1 (correct CWD) then train in section 5."
    )

arabic_dir = ckpt_root / "arabic"
if not (arabic_dir / "config.json").exists():
    raise FileNotFoundError(
        f"Expected {arabic_dir}/config.json — training may not have completed."
    )


def _dir_has_model_weights(d: Path) -> bool:
    if not d.is_dir():
        return False
    for p in d.rglob("*"):
        if not p.is_file():
            continue
        n = p.name.lower()
        if n == "pytorch_model.bin":
            return True
        if n.endswith(".safetensors") and p.stat().st_size > 500_000:
            return True
    return False


if not _dir_has_model_weights(arabic_dir):
    raise FileNotFoundError(
        f"No large weight file under {arabic_dir} "
        "(expect model*.safetensors or pytorch_model.bin). Wait for training to finish / Drive sync."
    )

print("Checkpoint tree (large files > 0.5 MB):")
for p in sorted(ckpt_root.rglob("*")):
    if p.is_file() and p.stat().st_size > 500_000:
        print(f"  {p.relative_to(ckpt_root)}  ({p.stat().st_size / 1e6:.1f} MB)")

zip_base = "/content/fndarija_checkpoints_bundle"
shutil.make_archive(zip_base, "zip", str(ROOT), "checkpoints")
zip_path = zip_base + ".zip"
print("ZIP:", zip_path, "—", f"{Path(zip_path).stat().st_size / 1e6:.1f} MB")
files.download(zip_path)

---

**Session tips:** Long training can disconnect — use **`TRAIN_MODE = "hub_demo"`** or **`"curated"`** first; switch to **`hub_full`** when stable. Keep the browser tab open. If Drive sync lags, wait until large files finish uploading before downloading the ZIP.

**Paths on PC after download:** unzip so `checkpoints/arabic/config.json` sits next to `app/` and `configs/` (same layout as this repo).